# Olist E-Commerce Analysis


## Objetivo
Analisar os dados de vendas para identificar concentração de receita, padrões ao longo do tempo e desempenho por região, gerando insights acionáveis para o negócio.

## IMPORTS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


## DATA LOADING

In [ ]:
df = pd.read_csv("../data/processed/final_dataset.csv")
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["month"] = df["order_purchase_timestamp"].dt.to_period("M")

df.head()

## DATA OVERVIEW

In [ ]:
df.info()
df.describe()

# ANÁLISES

## 📈 1. RECEITA AO LONGO DO TEMPO

In [ ]:
# Revenue over time
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["month"] = df["order_purchase_timestamp"].dt.to_period("M")

monthly_revenue = (
    df.groupby(["month", "order_id"])["payment_value"]
    .sum()
    .reset_index()
    .groupby("month")["payment_value"]
    .sum()
    .sort_index()
)

growth = monthly_revenue.pct_change() * 100

avg_growth = growth.mean()
volatility = growth.std()

plt.figure(figsize=(10,5))
monthly_revenue.plot()

peak_idx = monthly_revenue.idxmax()
plt.scatter(peak_idx, monthly_revenue[peak_idx])
plt.text(peak_idx, monthly_revenue[peak_idx],
         f"Peak\nR$ {monthly_revenue[peak_idx]:,.0f}",
         fontsize=9, ha='center', va='bottom')

plt.title("Revenue Over Time")
plt.xlabel("month")

plt.tight_layout()
plt.show()


## 📈 Dinâmica de Receita — Crescimento, Volatilidade e Previsibilidade

### Insight

- Crescimento médio mensal: **42,53%**
- Volatilidade (desvio padrão): **199,39%**
- Maior crescimento mensal: **+9569,82%**
- Maior queda mensal: **-99,98%**

### Interpretação

A volatilidade da receita é ~4,7x superior ao crescimento médio, indicando instabilidade estrutural.

### Implicação de Negócio

- Baixa previsibilidade de receita  
- Dependência de eventos pontuais  

### Ação Recomendada

- Identificar e replicar drivers de crescimento  
- Reduzir volatilidade com estratégias recorrentes  

# 📊 2. RECEITA POR CATEGORIA

In [ ]:
revenue_by_category = (
    df.groupby("product_category_name")["payment_value"]
    .sum()
    .sort_values(ascending=False)
)

top10 = revenue_by_category.head(10)
total = revenue_by_category.sum()

plt.figure(figsize=(10,5))
top10.sort_values().plot(kind="barh")

for i, v in enumerate(top10.sort_values()):
    pct = (v / total) * 100
    plt.text(v, i, f"{pct:.1f}%", va='center', fontsize=8)

plt.title("Top Categories by Revenue")
plt.xlabel("Revenue (R$)")

plt.tight_layout()
plt.show()

## 📊 Concentração de Receita — Dependência do Portfólio

### Insight

- Top 5 categorias: **38,97% da receita total**
- Top 10 categorias: **63,68% da receita total**

### Interpretação

Alta concentração de receita em poucas categorias.

### Implicação de Negócio

- Crescimento rápido possível  
- Alto risco de dependência  

### Ação Recomendada

- Maximizar categorias líderes  
- Desenvolver categorias secundárias  

# 🌍 3. RECEITA POR ESTADO

In [ ]:
revenue_by_state = (
    df.groupby("customer_state")["payment_value"]
    .sum()
    .sort_values(ascending=False)
)

top_states = revenue_by_state.head(10)
total = revenue_by_state.sum()

plt.figure(figsize=(10,5))
top_states.sort_values().plot(kind="barh")

for i, v in enumerate(top_states.sort_values()):
    pct = (v / total) * 100
    plt.text(v, i, f"{pct:.1f}%", va='center', fontsize=8)

plt.title("Top States by Revenue")
plt.xlabel("Revenue (R$)")

plt.tight_layout()
plt.show()

## 🌍 Concentração Regional — Estrutura de Mercado

### Insight

- SP representa ~37% da receita  
- Top 3 estados representam ~63%  

### Interpretação

Forte concentração geográfica.

### Implicação de Negócio

- Risco regional  
- Limitação de crescimento  

### Ação Recomendada

- Expandir para novos estados  
- Otimizar regiões principais  

# 💰 4. TICKET MÉDIO

In [ ]:
total_revenue = df["payment_value"].sum()
total_orders = df["order_id"].nunique()

avg_ticket = total_revenue / total_orders

plt.figure(figsize=(6,4))
plt.bar(["Average Ticket"], [avg_ticket])

plt.text(0, avg_ticket,
         f"R$ {avg_ticket:.2f}",
         ha='center', va='bottom')

plt.title("Average Ticket")

plt.tight_layout()
plt.show()

## 💰 Eficiência de Monetização — Ticket Médio

### Insight

- Receita total: **R$ 20,3M**
- Pedidos: **98.665**
- Ticket médio: **R$ 205,83**

### Interpretação

Ticket médio consistente com potencial de crescimento.

### Implicação de Negócio

- Crescimento via volume ou ticket  

### Ação Recomendada

- Upsell  
- Cross-sell  
- Bundles  

# ⭐ 5. REVIEWS VS RECEITA

In [ ]:
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

df_reviews = df.merge(reviews, on="order_id")

rating_revenue = df_reviews.groupby("review_score")["payment_value"].mean()

plt.figure(figsize=(8,4))
rating_revenue.sort_index().plot(kind="bar")

# destacar contraste 1 vs 5
diff = ((rating_revenue[1] - rating_revenue[5]) / rating_revenue[5]) * 100

plt.text(0, rating_revenue[1],
         f"+{diff:.1f}% vs 5★",
         ha='center', fontsize=9)

plt.title("Revenue by Review Score")
plt.xlabel("Review Score")

plt.tight_layout()
plt.show()

## ⭐ Qualidade vs Receita — Risco de Sustentabilidade

### Insight

- 1 estrela: **R$ 242,85**
- 5 estrelas: **R$ 167,75**

### Interpretação

Produtos com pior avaliação geram maior receita média.

### Implicação de Negócio

- Possível problema em produtos de alto valor  
- Risco de insatisfação do cliente  

### Ação Recomendada

- Melhorar qualidade dos produtos premium  
- Investigar causas de baixa avaliação  

---


# 📊 Executive Summary — Olist Sales Performance Analysis

---

## 🧭 Contexto

Esta análise tem como objetivo avaliar o desempenho de vendas da Olist, identificando padrões de crescimento, concentração de receita e possíveis riscos operacionais, com foco em geração de insights acionáveis para o negócio.

---

## 📈 1. Crescimento com Baixa Previsibilidade

A receita apresenta crescimento médio mensal de **42,53%**, porém com volatilidade extremamente elevada (**199,39%**), indicando um modelo de crescimento instável.

Esse comportamento sugere que o desempenho financeiro é impulsionado por **eventos pontuais**, e não por uma trajetória consistente de expansão.

### Impacto

- Dificuldade em prever receita  
- Risco elevado no planejamento financeiro  
- Dependência de picos de performance  

---

## 📊 2. Forte Dependência de Categorias Específicas

A análise de portfólio mostra que:

- **63,68% da receita** está concentrada em apenas 10 categorias  
- As categorias líderes são responsáveis pela maior parte do faturamento  

### Impacto

- Alta eficiência para crescimento no curto prazo  
- Risco estrutural de dependência  

---

## 🌍 3. Concentração Geográfica da Receita

A receita está altamente concentrada em regiões específicas:

- **São Paulo representa ~37% da receita total**  
- Os 3 principais estados concentram cerca de **63% da receita**

### Impacto

- Forte dependência de mercados maduros  
- Potencial limitado de crescimento se não houver expansão  

---

## 💰 4. Monetização Consistente com Potencial de Expansão

O ticket médio atual é de **R$ 205,83**, indicando um bom nível de monetização por pedido.

### Impacto

- Possibilidade de crescimento via aumento de volume  
- Oportunidade clara de elevar ticket médio com estratégias comerciais  

---

## ⚠️ 5. Risco Crítico: Receita vs Satisfação do Cliente

Um dos achados mais relevantes é a relação inversa entre avaliação do cliente e receita:

- Produtos com **1 estrela geram maior receita média (R$ 242,85)**  
- Produtos com **5 estrelas geram menor receita média (R$ 167,75)**  

### Interpretação

Esse padrão indica um possível desalinhamento entre **preço, expectativa e entrega de valor**, especialmente em produtos de maior ticket.

### Impacto

- Risco de insatisfação e churn  
- Possível deterioração da reputação da marca  
- Impacto negativo na sustentabilidade do crescimento  

---

## 🧠 Diagnóstico Geral

O negócio apresenta:

✔ Forte geração de receita  
✔ Capacidade de crescimento acelerado  
✔ Categorias bem estabelecidas  

Porém com riscos estruturais relevantes:

⚠ Alta volatilidade  
⚠ Concentração excessiva (produto + região)  
⚠ Possível problema de qualidade em produtos de maior valor  

---

## 🚀 Recomendações Estratégicas

### Curto Prazo (Ganho Rápido)

- Replicar fatores que impulsionaram picos de receita  
- Intensificar investimento nas categorias líderes  
- Otimizar performance nas principais regiões  

---

### Médio Prazo (Sustentabilidade)

- Diversificar portfólio de categorias  
- Expandir atuação geográfica  
- Implementar estratégias para aumentar previsibilidade  

---

### Longo Prazo (Escala)

- Corrigir desalinhamento entre preço e experiência do cliente  
- Estruturar crescimento baseado em consistência, não em picos  
- Monitorar satisfação como métrica estratégica  

---

## 📌 Conclusão

A Olist apresenta um modelo de crescimento com alto potencial, porém sustentado por variáveis instáveis e concentradas.

A capacidade de transformar esse crescimento em algo previsível e escalável dependerá diretamente da **redução de volatilidade, diversificação de receita e melhoria da experiência do cliente**.